In [1]:
from torchvision import models, transforms
import torch.nn as nn

In [2]:
# 1. THE PREP COOK (Transforms)
data_transforms = transforms.Compose([
    transforms.Resize(256),            # Resize smaller side to 256
    transforms.CenterCrop(224),        # Crop the center to exactly 224x224
    transforms.ToTensor(),             # Convert 0-255 pixels to 0.0-1.0 tensors
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) # Standard ImageNet magic numbers
])

In [3]:
# 2. THE MODEL (ResNet-18)
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Freeze layers
for param in model.parameters():
    param.requires_grad = False

In [5]:
# 3. THE SURGERY (Replace the head for 3 classes)
model.fc = nn.Linear(model.fc.in_features, 3)

print("Model is ready for training on 3 classes!")

Model is ready for training on 3 classes!


In [6]:
def predict_flower(image_path, model):
    # 1. Prep the image (Crop, Resize, Normalize)
    img = load_image(image_path)
    img_tensor = data_transforms(img) # Using the transforms from earlier
    
    # 2. Add the Batch Dimension
    # From [3, 224, 224] -> [1, 3, 224, 224]
    input_batch = img_tensor.unsqueeze(0)
    
    # 3. Put model in 'evaluation' mode 
    # This turns off things like BatchNorm that are only for training
    model.eval()
    
    # 4. Predict!
    with torch.no_grad():
        output = model(input_batch)
    
    # 5. Get the highest score
    # output is a list of 'scores'. The highest score is our winner.
    _, predicted_class = torch.max(output, 1)
    
    return predicted_class.item()